# Название эксперимента (одной строкой, по-русски)

| Поле | Значение |
|------|----------|
| **Автор** | Имя Фамилия |
| **Дата** | ГГГГ-ММ-ДД |
| **Гипотеза** | Г1 (ЭЭГ) / Г2 (МРТ) |
| **Цель** | Что проверяем и какой результат будет считаться успехом (1–2 предложения) |
| **Данные** | Набор, число пациентов по классам, источник, версия. Классы из разных источников не смешивать |
| **Модель** | Название, откуда веса (ссылка), дообучали или нет |
| **Деление обучение/тест** | **По пациентам** (id пациента — один и тот же во всех его кусках). Как именно: hold-out 80/20 / 5-fold по пациентам; seed |
| **Среда** | Kaggle (T4) / Ubuntu / Windows; версии ключевых библиотек |
| **Журнал** | Строка в `experiments_log.md` добавлена: да / нет |

> Блокнот — это отчёт: перед коммитом выполните **Restart kernel and Run All**, выводы ячеек оставьте.

In [ ]:
# 0. Настройка среды. Эту ячейку не менять (кроме SEED).
import sys, platform
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir()), None)
if ROOT is None and Path("/kaggle/working/epilepsy-mvp").exists():
    ROOT = Path("/kaggle/working/epilepsy-mvp")      # Kaggle: репозиторий склонирован сюда
assert ROOT is not None, "Не найден корень репозитория (папка common/). См. README, раздел «Kaggle»."
sys.path.insert(0, str(ROOT))

from common.paths import data_dir, weights_dir, platform_name
from common.splits import patient_holdout, patient_kfold, assert_no_leak
from common.metrics import aggregate_by_patient, binary_report, bootstrap_auc_ci

SEED = 42
print("Платформа:", platform_name(), "| Python", platform.python_version())
print("Данные:", data_dir())

## 1. Данные
Откуда загружаем, сколько пациентов и записей в каждом классе, длина кусков, частота дискретизации / вокселя.
Обязательно: таблица «пациент → метка → число кусков» и проверка, что у одного пациента одна метка.

In [ ]:
# Загрузка данных и сводная таблица по пациентам

## 2. Предобработка
Фильтры, передискретизация, выбор каналов / нормализация интенсивности, регистрация. Что и зачем; параметры вынести в переменные в начале ячейки.

In [ ]:
# Предобработка

## 3. Модель
Какая модель, какие веса (путь к весам — вне git, через `weights_dir()`), как обучаем / используем готовую. Деление по пациентам — здесь.

In [ ]:
# Деление по пациентам (пример):
# train_idx, test_idx = patient_holdout(groups=patient_ids, test_size=0.2, seed=SEED)
# assert_no_leak(patient_ids[train_idx], patient_ids[test_idx])

# Модель, обучение / предсказание

## 4. Метрики
Считаем **на уровне пациента** (`aggregate_by_patient`), с доверительным интервалом (`bootstrap_auc_ci`). Порог выбирается на валидации, не на тесте.
Графики: ROC-кривая, распределение оценок по классам. Для картинок `dpi=100`, чтобы блокнот не раздувался.

In [ ]:
# Метрики и графики
# df = aggregate_by_patient(patient_ids[test_idx], y[test_idx], scores)
# print(binary_report(df.y, df.score)); print(bootstrap_auc_ci(df.y, df.score))

## 5. Выводы
- Главный результат одной строкой с цифрами и ДИ.
- Сравнение с ориентиром (для Г1: ROC AUC ~0,76–0,79).
- Ограничения: мало пациентов, возможная утечка, смещение по источнику данных.
- Что нельзя утверждать по этим данным.

## 6. Что дальше
- Следующий эксперимент (1–3 пункта) и что для него нужно (данные, GPU, доступ).